<img src="https://keystoneacademic-res.cloudinary.com/image/upload/c_pad,w_640,h_304/dpr_auto/f_auto/q_auto/v1/element/94/94774_thumb.png" width=300>

# Programação para Análise de Dados

## Aula 3: Fundamentos do pandas e leitura de dados. 

### Professor: Carlos E. Leal de Castro

### ANTES!

- Sentem nos mesmos computadores que vocês configuraram na aula passada! Liguem os Computadores! Façam o login!

- Se for o seu computador ou o mesmo, o processo de criação do nosso Env já deveria estar instalado!

- Abram (ou baixem) o Visual Code Studio! Vocês vão acompanhar a aula por lá!

- Caso queiram, abram o Prompt do Anaconda e digitem:

```bash
conda activate kernelPAD && jupyter notebook
```

### Objetivos de Aprendizagem
- Compreender as estruturas **Series** e **DataFrame** do `pandas`.
- Ler dados de arquivos **CSV** e **Excel**.
- Realizar **exploração inicial** dos dados (`head`, `tail`, `info`, `describe`).
- Executar **seleção e filtragem** básica (`[]`, `.loc`, `.iloc`, máscaras booleanas e `query`).

> Slides marcados como **EXERCÍCIO** devem ser feitos em sala

## Aquecimento: o que é transformação de dados?

Transformação de dados é o processo de **organizar, limpar e padronizar** dados para análise.  
Nesta aula, focaremos no **primeiro contato**: carregar, inspecionar e fazer seleções simples.

## Setup do Ambiente

In [ ]:
import pandas as pd
import numpy as np

# Caminho para os dados usados nos exercícios (ajuste se necessário)
DATA_DIR = "."
#CLIENTES_CSV = "dados_clientes_ptbr.csv"
CLIENTES_CSV = 'https://raw.githubusercontent.com/DeepFluxion/2026_2_IBMEC_PROG_ANALISE_DADOS/refs/heads/main/datasets/dados_clientes_ptbr.csv'
#VENDAS_CSV   = "dados_vendas.csv"
VENDAS_CSV   = "https://raw.githubusercontent.com/DeepFluxion/2026_2_IBMEC_PROG_ANALISE_DADOS/refs/heads/main/datasets/dados_vendas.csv"

## Estruturas de dados do pandas

**Series** é um vetor rotulado unidimensional.  
**DataFrame** é uma tabela bidimensional com rótulos (linhas/colunas).

Características importantes:
- Rótulos (`index` e `columns`) para alinhamento automático.
- Operações **vetorizadas** (rápidas) e funções convenientes.
- Suporte a tipos heterogêneos por coluna.

### Criando uma `Series`

In [ ]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="minha_serie")
print(s)

### Criando um `DataFrame`

In [ ]:
dados = {
    "produto": ["A", "B", "C", "D", "E"],
    "preco": [10.0, 15.5, 8.7, 42.0, 66.6],
    "estoque": [100, 50, 0, 5, 13],
}
df = pd.DataFrame(dados)
df

### Atributos úteis

In [ ]:
df.shape, df.dtypes, df.index, df.columns

## Leitura de Arquivos CSV

#### Dica rápida
- CSV em pt-BR muitas vezes usa **;** como separador e **,** como separador decimal.  
- Use `sep=';'` e `decimal=','`.  
- Caso seu arquivo esteja em `latin-1`/`ISO-8859-1`, use `encoding='latin-1'`.

In [ ]:
# Lendo um CSV pt-BR (fornecido com este notebook)
df_clientes = pd.read_csv(CLIENTES_CSV, sep=';', decimal=',', encoding='utf-8')
df_clientes.tail(2)

In [ ]:
# Parâmetros essenciais do read_csv (demonstração)
amostra = pd.read_csv(
    CLIENTES_CSV,
    sep=';',
    decimal=',',
    usecols=['id', 'nome', 'cidade', 'idade', 'renda_mensal'],
    dtype={'id': 'int64', 'idade': 'Int64'},
    nrows=3,
#     parse_dates=['data_cadastro'],  # Ignorado aqui porque o CSV tem data dd/mm/yyyy como texto
    dayfirst=True,                  # útil para dd/mm/yyyy quando parse_dates estiver ativo
    engine='python'                 # útil em arquivos complexos
)
amostra

In [ ]:
# Ajuste de datas quando o parse_dates não funcionar direto (ex: dd/mm/yyyy em coluna texto)
df_clientes['data_cadastro'] = pd.to_datetime(df_clientes['data_cadastro'],
                                              format='%d/%m/%Y')

df_clientes['coluna_teste'] = [1,2,3,4,5]
# df_clientes.dtypes
df_clientes.head()

## Leitura de Arquivos Excel (XLS/XLSX)

Para ler `.xlsx`, instale o motor adequado (ex.: `openpyxl`).  
Ex.: `pip install openpyxl`

Parâmetros úteis: `sheet_name`, `usecols`, `skiprows`, `nrows`, `dtype`, `parse_dates`.

In [ ]:
!pip install openpyxl 
# EXECUTE

In [ ]:
# Exemplo (ajuste o caminho para um arquivo .xlsx que você possua):
# df_excel = pd.read_excel("meu_arquivo.xlsx", sheet_name=0, usecols="A:F")
# df_excel.head()
print("Exemplo comentado: use pd.read_excel('arquivo.xlsx') quando tiver um arquivo Excel disponível.")

## Exploração Inicial dos Dados

Para ler seus dados no Google Drive, faça o seguinte:`
- Seu link de compartilhamento, provavelmente, é assim: `https://drive.google.com/file/d/1kQmDMkMvQwYjvcG9Y52AfPm-6KTIZWKP/view?usp=drive_link`
- O ID do arquivo é: `1kQmDMkMvQwYjvcG9Y52AfPm-6KTIZWKP`

Com isso, basta montar o link assim:

- `https://drive.google.com/uc?export=download&id=ID_DO_ARQUIVO`

In [ ]:
# ID do arquivo no Google Drive
file_id = "1kQmDMkMvQwYjvcG9Y52AfPm-6KTIZWKP"
# Link direto de download
url = f"https://drive.google.com/uc?export=download&id={file_id}"
# Lendo o CSV
df = pd.read_csv(url)
# Mostrando as 5 primeiras linhas
df.head()

In [ ]:
df.tail(3) # Mostra as ultimas 3 linhas

In [ ]:
df.info()  # tipos, nulos e memória

In [ ]:
df.describe()  # estatísticas de colunas numéricas

In [ ]:
# Outras explorações úteis
df['artist_name'].value_counts()

## Seleção e Filtragem Básica (25 min)

### Selecionando colunas e linhas
- `df['col']` ou `df[['col1','col2']]`
- `.loc[linhas, colunas]` por **rótulo**
- `.iloc[linhas, colunas]` por **posição**

In [ ]:
# Colunas
df['artist_name'].head()

In [ ]:
df[['artist_name', 'popularity']].head()

In [ ]:
# Linhas por rótulo vs posição
df.loc[0:8, ['artist_name', 'popularity', 'danceability']]

In [ ]:
df.iloc[0:3, 0:5]

### Filtros com máscaras booleanas e `query`

In [ ]:
# Máscara booleana
filtro = (df['popularity'] >= 50) & (df['instrumentalness'] < 0.5)
df[filtro][['genre','popularity','energy','speechiness','instrumentalness']]

In [ ]:
df.columns

In [ ]:
# query (requer nomes de colunas "seguros", sem espaços)
# Vamos criar uma cópia com colunas "seguras"
artistas = df.rename(columns={'artist_name': 'artist_name_'}).copy()
# Para usar query com números, primeiro convertemos a renda para float padrão.
artistas['artist_name_'] = artistas['artist_name_'].str.replace(' ', '_')
artistas.query("popularity >= 30 and energy > 0.5")[['artist_name_','genre','popularity','energy','speechiness']]

### Ordenação e amostras

In [ ]:
df.sort_values(by=['popularity', 'artist_name'], ascending=[False, True]).head()

## Selecionar Amostras aleatórias do dataset

In [ ]:
df.sample(3, random_state=42)

### Agrupamento de Dados com `groupby` no pandas

#### O que é `groupby`?
- É um método do pandas usado para **agrupar linhas** com base em uma ou mais colunas.
- Permite aplicar funções de agregação (`sum`, `mean`, `count`, etc.) para cada grupo.

##### Analogia
Pense em `groupby` como:
1. **Dividir** os dados em grupos com base em um critério.
2. **Aplicar** uma função a cada grupo.
3. **Combinar** os resultados em um novo DataFrame ou Series.

### Sintaxe Básica
```python
df.groupby('coluna').funcao_agregacao()
df.groupby(['coluna1', 'coluna2']).funcao_agregacao()
```

### Média de popularidade por gênero

In [ ]:
df.groupby("genre")["popularity"].mean().sort_values(ascending=False).head(10)

### Desvio padrão de Artista por Dançabilidade

In [ ]:
df.groupby("artist_name")["danceability"].std().sort_values(ascending=False).head(10)

### Média de Gênero por Loudness e Speechiness

In [ ]:
df.groupby("genre")[["loudness", "speechiness"]].mean().sort_values("genre", ascending=False).head(10)

### Popularidade média por gênero e modo (Major/Minor)

In [ ]:
df.groupby(["genre", "mode"])["popularity"].mean().reset_index()

## Exercícios — Introdução ao Pandas
1. Leitura e visualização inicial

    - Leia o arquivo melb_data.csv usando pandas.read_csv().

    - Mostre as 5 primeiras linhas do DataFrame.

2. Estatísticas descritivas

    - Mostre as estatísticas descritivas (describe()) apenas para as colunas Price e BuildingArea.

3. Filtro por bairro

    - Filtre apenas os imóveis localizados no CouncilArea "Melbourne".

    - Mostre as colunas Adress, Type e Price.

4. Ordenação

    - Ordene o DataFrame pelo preço (Price) em ordem decrescente.

    - Mostre apenas as 10 primeiras linhas.

5. Agrupamento

    - Agrupe os dados por ano de construção ('YearBuilt') e calcule o preço médio para cada ano.

## Referências e Leituras Recomendadas

- Documentação oficial do pandas (User Guide)  
- Leitura de dados: `read_csv`, `read_excel`, e guia de IO tools  
- Indexação/seleção: `.loc`, `.iloc`, máscaras, `query`  
- Funções básicas/essenciais